# 1、多个中间件组合及执行顺序

问题：Middleware 可以叠加使用，那么多个中间件书写顺序重要吗？

非常重要！





In [1]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

比如：

In [ ]:
from fastmcp.server.middleware.logging import LoggingMiddleware
from langchain.agents.middleware import SummarizationMiddleware, TrimmerMiddleware

middleware=[
    TrimmerMiddleware(),       # 1. 先修剪消息（截断超长消息、控制上下文长度）
    SummarizationMiddleware(), # 2. 再对历史对话生成摘要压缩上下文
    LoggingMiddleware()        # 3. 最后记录完整处理后的日志
]

比如：

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, PIIMiddleware, ToolRetryMiddleware

agent = create_agent(
    model=model,
    tools=[get_weather, get_news],
    middleware=[
        PIIMiddleware(strategy="redact"),                      # 1. 最先检测并脱敏敏感信息
        ModelCallLimitMiddleware(run_limit=10),                 # 2. 限制单次运行模型调用次数
        SummarizationMiddleware(max_tokens_before_summary=500), # 3. 上下文token达到阈值自动摘要压缩历史
        ToolRetryMiddleware(max_retries=3),                     # 4. 工具执行异常自动重试
    ]
)

### 举例：

In [2]:
from langchain.agents.middleware import AgentMiddleware
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

class Middleware1(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件1] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件1] after_model")
        return None

class Middleware2(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件2] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件2] after_model")
        return None

class Middleware3(AgentMiddleware):
    def before_model(self, state, runtime):
        print("[中间件3] before_model")
        return None
    def after_model(self, state, runtime):
        print("[中间件3] after_model")
        return None

agent = create_agent(
    model=model,
    tools=[],
    middleware=[Middleware1(), Middleware2(), Middleware3()]
)
print("\n执行一次调用，观察顺序：")
agent.invoke({"messages": [HumanMessage("测试")]})

print("\n==== 执行顺序关键点 ====")
print(" - before_model：正序执行 1 → 2 → 3")
print(" - after_model：逆序执行 3 → 2 → 1")
print(" - 洋葱模型完整链路：1.before → 2.before → 3.before → 大模型调用 → 3.after → 2.after → 1.after")


执行一次调用，观察顺序：
[中间件1] before_model
[中间件2] before_model
[中间件3] before_model
[中间件3] after_model
[中间件2] after_model
[中间件1] after_model

==== 执行顺序关键点 ====
 - before_model：正序执行 1 → 2 → 3
 - after_model：逆序执行 3 → 2 → 1
 - 洋葱模型完整链路：1.before → 2.before → 3.before → 大模型调用 → 3.after → 2.after → 1.after
